In [1]:
import os
import glob
import yaml
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import sys
sys.path.append('..') # src 폴더 인식을 위한 경로 추가
from src import build_parent_retriever, run_corrective_agent

In [22]:
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

✅ Gemini API 키 로드 완료


In [2]:
data_dir = "../data/raw/"
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))

print(f"📂 {len(pdf_files)}개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...")

# src/document_processor.py 에 정의된 함수 단 한 줄로 복잡한 DB 구축 끝!
retriever, all_docs = build_parent_retriever(pdf_files, chunk_size=400)

📂 1개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...
⏳ 1개의 문서를 로드하고 Vector DB를 구축합니다...


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33166.43it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:30: La

✅ 다중 문서 기반 하이브리드 DB 구축 완료!


In [6]:
# ==========================================
# 🧠 동적 에이전트 루프 실행 (다중 후보 순회 탐색 적용)
# ==========================================
print("\n" + "="*50)
print("🧐 다중 문서 환경에서의 하이브리드 에이전트 테스트")

query = "삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# --- Step 1: 텍스트 기반 후보군 탐색 ---
print("🔍 [Path A] 10개 기업의 수만 개 조각 중 목표 위치 후보들을 탐색합니다...")
retrieved_docs = retriever.invoke(query)

if not retrieved_docs:
    print("❌ 관련된 문서를 찾지 못했습니다.")
else:
    print(f"🎯 총 {len(retrieved_docs)}개의 유력한 후보 페이지 묶음을 발견했습니다!\n")
    final_answer = ""
    
    # --- Step 2: 후보 페이지들을 순서대로 돌면서 에이전트 가동 ---
    for i, doc in enumerate(retrieved_docs):
        target_pdf = doc.metadata.get('source')
        start_page = doc.metadata.get('page', 0)
        
        print("-" * 40)
        print(f"▶️ [후보 탐색 {i+1}/{len(retrieved_docs)}] [{os.path.basename(target_pdf)}]의 {start_page + 1}페이지 주변 탐색 시작...")
        
        # 에이전트 가동
        answer = run_corrective_agent(query, target_pdf, start_page, all_docs, max_expansions=2)
        
        # 🛑 정답 도출 여부 판별 (실패 시 LLM이 보통 내뱉는 문구로 필터링)
        # 만약 프롬프트에 "정답이 없으면 '정보없음' 이라고 출력해" 라고 추가해두면 더 정확합니다.
        if "포함되어 있지 않습니다" in answer or "찾기 어렵습니다" in answer or "제시되어 있지 않습니다" in answer:
            print("⏭️ AI 평가: '현재 후보군에는 명확한 수치가 없습니다.' -> 다음 후보 위치로 이동합니다.")
            continue # 다음 검색된 페이지로 넘어감
        else:
            print("✅ AI 평가: '완벽한 정답을 찾았습니다!' -> 탐색을 조기 종료합니다.")
            final_answer = answer
            break # 정답을 찾았으므로 반복문 종료!

    # 모든 후보를 돌았는데도 못 찾았을 경우의 안전장치
    if not final_answer:
        final_answer = "검색된 모든 후보 페이지(N, N+1, N+2)를 탐색했으나 명확한 수치를 찾지 못했습니다."

    print("\n" + "=" * 50)
    print("🏆 최종 AI 답변 (Iterative Corrective RAG):")
    print(final_answer)
    print("=" * 50)


🧐 다중 문서 환경에서의 하이브리드 에이전트 테스트
질문: 삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?

🔍 [Path A] 10개 기업의 수만 개 조각 중 목표 위치 후보들을 탐색합니다...
🎯 총 2개의 유력한 후보 페이지 묶음을 발견했습니다!

----------------------------------------
▶️ [후보 탐색 1/2] [samsung_report.pdf]의 81페이지 주변 탐색 시작...

🔄 [Agent Loop 1] 분석 중인 페이지: [81, 82, 83]
🤔 Gemini 2.5 Flash가 다중 모달 데이터를 교차 검증 중입니다...
🚨 AI 평가: '표가 더 이어집니다.' -> 윈도우를 다음 페이지로 동적 확장합니다.

🔄 [Agent Loop 2] 분석 중인 페이지: [81, 82, 83, 84]
🤔 Gemini 2.5 Flash가 다중 모달 데이터를 교차 검증 중입니다...
🚨 AI 평가: '표가 더 이어집니다.' -> 윈도우를 다음 페이지로 동적 확장합니다.

🔄 [Agent Loop 3] 분석 중인 페이지: [81, 82, 83, 84, 85]
🤔 Gemini 2.5 Flash가 다중 모달 데이터를 교차 검증 중입니다...
✅ AI 평가: '현재 문맥 내에 완벽한 정답이 존재합니다.' -> 탐색 완료.
⏭️ AI 평가: '현재 후보군에는 명확한 수치가 없습니다.' -> 다음 후보 위치로 이동합니다.
----------------------------------------
▶️ [후보 탐색 2/2] [samsung_report.pdf]의 293페이지 주변 탐색 시작...

🔄 [Agent Loop 1] 분석 중인 페이지: [293, 294, 295]
🤔 Gemini 2.5 Flash가 다중 모달 데이터를 교차 검증 중입니다...
🚨 AI 평가: '표가 더 이어집니다.' -> 윈도우를 다음 페이지로 동적 확장합니다.

🔄 [Agent Loop 2] 분석 중인 페이지: [2